# R-CNN training

In [98]:
import selectivesearch

# Standard libraries
from pathlib import Path
import os
import random
import time
import re
from pathlib import Path
from collections import defaultdict, Counter
from itertools import islice

# Image processing libraries
import cv2
import imageio.v3 as imageio
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torchvision.transforms import functional as TF

# Machine learning and deep learning libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as func
import torchvision.models as models
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Data management libraries
import pandas as pd
import json
import orjson
import shutil 
import pickle

# Parallelism and multiprocessing libraries
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

# Progress and monitoring libraries
from tqdm import tqdm

# Dataset management libraries
from torch.utils.data import Dataset, DataLoader

# PyTorch models and transformations libraries
from torchvision import transforms

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import torchvision
from torch.optim.lr_scheduler import ReduceLROnPlateau

import matplotlib.patches as patches
import torch.nn.utils as utils
import ast
import itertools

## Setup

In [86]:
# Processed data path

PROCESSED_DATA_PATH = Path("../data/processed")
MODELS_PATH = Path("../models")

image_folder_path = PROCESSED_DATA_PATH / "images"

# COCO json paths
coco_json_path = PROCESSED_DATA_PATH / "coco_annotations_new.json"
original_coco_json_path = PROCESSED_DATA_PATH / "coco_annotations.json"
new_coco_json_path = PROCESSED_DATA_PATH / "mod_coco_annotations.json"

# Proposals
proposal_folder_path = PROCESSED_DATA_PATH / "active_regions_dataset" / "proposals"
proposal_json_path = PROCESSED_DATA_PATH / "active_regions_dataset" / "proposals.json"
output_proposal_json_path = PROCESSED_DATA_PATH / "active_regions_dataset" / "proposals.json"
in_new_coco_json_path = proposal_folder_path / "mod_coco_annotations.json"

# Active regions
active_region_folder_path = PROCESSED_DATA_PATH / "active_regions_dataset"
actproposal_json_path = active_region_folder_path / "active_regions.json"

# Train/Val/Test splits
train_path = PROCESSED_DATA_PATH / "active_regions_dataset" / "train.json"
val_path = PROCESSED_DATA_PATH / "active_regions_dataset" / "val.json"
test_path = PROCESSED_DATA_PATH / "active_regions_dataset" / "test.json"

# Zip file path
zip_active_regions = active_region_folder_path / "activeregion-xview-dataset.zip"
zip_proposals = active_region_folder_path / "proposals-dataset.zip"
zip_full_dataset = PROCESSED_DATA_PATH / "our-xview-dataset.zip"

# Others
class_map_path = PROCESSED_DATA_PATH / "xView_class_map.json"
labels_parquet_path = PROCESSED_DATA_PATH / "xView_labels.parquet"

# Models path
alexnet_dir = MODELS_PATH / "AlexNet"
os.makedirs(alexnet_dir, exist_ok=True)
alexnet_path = alexnet_dir / "AlexNet_best.pth"
resnet_dir = MODELS_PATH / "ResNet50"
os.makedirs(resnet_dir, exist_ok=True)
resnet_path = resnet_dir / "ResNet50_best.pth"

regressor_path = resnet_dir / "regressor_best.pth"
regressor_input_path = actproposal_json_path
regressor_output_path = resnet_dir / "regressed_boxes.json"


RANDOM_SEED = 42

## Utility

In [3]:
import warnings

# Suppress specific warnings from the skimage module
warnings.filterwarnings("ignore", 
    message="Applying `local_binary_pattern` to floating-point images may give unexpected results.*")

# Suppress warnings related to the deprecated 'pretrained' parameter
warnings.filterwarnings("ignore", 
    message="The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.")

# Suppress warnings related to arguments other than a weight enum or None
warnings.filterwarnings("ignore", 
    message="Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future.*")

# Suppress warnings related to the deprecated 'verbose' parameter
warnings.filterwarnings("ignore", 
    message="The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.")

# Suppress warnings related to `torch.load` and `weights_only=False`
warnings.filterwarnings("ignore", 
    message="You are using `torch.load` with `weights_only=False`.*")

In [4]:
def load_json(file_path):
    """
    Load a JSON file from the specified path.

    :param file_path: Path to the JSON file to load.
    :return: Data contained in the JSON file (as a dictionary or list).
    """
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

## Test the dataset

In [5]:
def display_images_with_bboxes(json_file, specific_images, images_folder):
    """
    Display the specified images with all bounding boxes overlaid.
    
    :param json_file: Path to the JSON file containing images, annotations, and categories
    :param specific_images: List of image names to display
    :param images_folder: Path to the folder containing the images
    """
    # Load the JSON file
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Extract images and annotations
    images = data["images"]
    annotations = data["annotations"]
    
    # Create a dictionary to map image IDs to file names
    image_dict = {image["id"]: image["file_name"] for image in images}
    
    # Filter annotations for the specific images
    specific_annotations = [ann for ann in annotations if image_dict[ann["image_id"]] in specific_images]

    # Create a dictionary to collect all annotations for each image
    image_bboxes = {}
    for annotation in specific_annotations:
        image_name = image_dict[annotation["image_id"]]
        if image_name not in image_bboxes:
            image_bboxes[image_name] = []
        bbox = annotation["bbox"]
        category_id = annotation["category_id"]
        image_bboxes[image_name].append((bbox, category_id))
    
    # Display all images with all bounding boxes
    for image_name, bboxes in image_bboxes.items():
        # Load the image
        image_path = f'{images_folder}/{image_name}'  # Use the correct path for the images
        image = Image.open(image_path)

        # Create the figure for visualization
        plt.figure(figsize=(8, 8))
        plt.imshow(image)

        # Add all bounding boxes and category_id
        for bbox, category_id in bboxes:
            x, y, w, h = ast.literal_eval(bbox)
            # Add the bounding box
            plt.gca().add_patch(plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none'))
            # Display the category_id near the bounding box (smaller)
            plt.text(x, y, str(category_id), color='blue', fontsize=8, ha='left', va='top', backgroundcolor='white')

        # Set the title and turn off the axes
        plt.title(f"Image: {image_name}")
        plt.axis('off')
        plt.show()

In [ ]:
specific_images = ['img_418_2240_3200.jpg', 'img_1051_1600_320.jpg']

display_images_with_bboxes(coco_json_path, specific_images, image_folder_path)

## COCO preprocessing

In [7]:
def process_custom_coco_json(input_path, output_path):
    """
    Processed json COCO in a custom format.
    """
    # Read the JSON from the input file
    data = load_json(input_path)

    # Get and correct the format of the categories
    raw_categories = data.get('categories', [])
    categories = []
 
    for category in tqdm(raw_categories, desc="Processing Categories"):
        for id_str, name in category.items():
            try:
                categories.append({"id": int(id_str), "name": name})
            except ValueError:
                print(f"Error parsing category: {category}")
 
    # Find the "Aircraft" category with ID 0
    aircraft_category = next((cat for cat in categories if cat['id'] == 0 and cat['name'] == "Aircraft"), None)
    if aircraft_category:
        aircraft_category['id'] = 11  # Change the ID of the "Aircraft" category to 11
 
    # Add the "background" category with ID 0 if it doesn't exist
    if not any(cat['id'] == 0 for cat in categories):
        categories.append({"id": 0, "name": "background"})
 
    # Preprocess the annotations into a dictionary by image
    image_annotations_dict = {}
    for annotation in tqdm(data.get('annotations', []), desc="Building Image Annotations Dictionary"):
        image_id = annotation['image_id']
        if image_id not in image_annotations_dict:
            image_annotations_dict[image_id] = []
        image_annotations_dict[image_id].append(annotation)
 
    # List of annotations to keep (only valid ones)
    valid_annotations = []
    annotations_to_remove = set()
 
    # Bounding box check
    for annotation in tqdm(data.get('annotations', []), desc="Processing Annotations"):
        if annotation['category_id'] == 0:  # If it's Aircraft
            annotation['category_id'] = 11
        
        # Convert the bbox format
        if isinstance(annotation['bbox'], str):
            annotation['bbox'] = json.loads(annotation['bbox'])
        
        x, y, width, height = annotation['bbox']
        xmin, xmax = x, x + width
        ymin, ymax = y, y + height
        
        # Check that xmin < xmax and ymin < ymax, and that the width and height are sufficient
        if xmin >= xmax or ymin >= ymax or width <= 10 or height <= 10:
            annotations_to_remove.add(annotation['id'])
        else:
            annotation['bbox'] = [xmin, ymin, xmax, ymax]
            valid_annotations.append(annotation)
 
    # Remove invalid annotations
    data['annotations'] = valid_annotations
 
    # Check if there are images without annotations (using the annotations dictionary)
    new_annotations = []
    for image in tqdm(data.get('images', []), desc="Processing Images"):
        if image['id'] not in image_annotations_dict:  # If the image has no annotations
            # Add the "background" category
            new_annotation = {
                'id': len(data['annotations']) + len(new_annotations),
                'image_id': image['id'],
                'category_id': 0,  # Background category with ID 0
                'area': image['width'] * image['height'],
                'bbox': [0.0, 0.0, image['width'], image['height']],  # Background with bbox covering the entire image
                'iscrowd': 0
            }
            new_annotations.append(new_annotation)
 
    # Add the new annotations to the original JSON
    data['annotations'].extend(new_annotations)

    # Update the categories in the JSON
    data['categories'] = categories

    # **Randomly select 50% of the images**
    all_images = data.get('images', [])
    selected_images = random.sample(all_images, len(all_images) // 2)
    selected_image_ids = {img['id'] for img in selected_images}

    # Filter images and annotations
    data['images'] = selected_images
    data['annotations'] = [ann for ann in data['annotations'] if ann['image_id'] in selected_image_ids]

    # Write the modified JSON to the output file
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=4)

In [ ]:
process_custom_coco_json(coco_json_path, new_coco_json_path)

Processing Images: 100%|██████████| 45891/45891 [00:00<00:00, 550686.22it/s]


## Category check

In [9]:
def count_bounding_boxes(json_path):
    """
    Count the number of bounding boxes for each category in a COCO JSON file.

    Args:
        json_path (str): Path to the JSON file.

    Returns:
        list: List of tuples with category ID, category name, and number of bounding boxes.
    """
    # Load the JSON file
    coco_data = load_json(json_path)

    # Extract main data
    annotations = coco_data.get("annotations", [])
    categories = coco_data.get("categories", [])

    # Map category IDs to category names
    category_id_to_name = {category["id"]: category["name"] for category in categories}

    # Count bounding boxes per category
    bbox_counts = defaultdict(int)
    for annotation in annotations:
        category_id = annotation["category_id"]
        bbox_counts[category_id] += 1

    # Create a list of results
    results = [
        (cat_id, category_id_to_name.get(cat_id, "Unknown"), count)
        for cat_id, count in bbox_counts.items()
    ]
    
    # Sort the results in descending order by the number of bounding boxes
    results.sort(key=lambda x: x[2], reverse=True)
    
    # Print the results
    for cat_id, category_name, count in results:
        print(f"Category ID {cat_id} ('{category_name}'): {count} bounding boxes")

In [10]:
count_bounding_boxes(coco_json_path)

Category ID 6 ('Building'): 171112 bounding boxes
Category ID 1 ('Passenger Vehicle'): 47720 bounding boxes
Category ID 2 ('Truck'): 11520 bounding boxes
Category ID 0 ('background'): 6833 bounding boxes
Category ID 4 ('Maritime Vessel'): 2448 bounding boxes
Category ID 9 ('Shipping Container'): 2377 bounding boxes
Category ID 5 ('Engineering Vehicle'): 2331 bounding boxes
Category ID 3 ('Railway Vehicle'): 1736 bounding boxes
Category ID 8 ('Storage Tank'): 889 bounding boxes
Category ID 11 ('Aircraft'): 770 bounding boxes
Category ID 10 ('Pylon'): 219 bounding boxes
Category ID 7 ('Helipad'): 66 bounding boxes


## Region proposals generation

In [11]:
# Function to generate region proposals with Selective Search
def generate_and_process_proposals(image, img_width, img_height):
    img_np = np.array(image, dtype=np.uint8)

    # Perform selective search with optimized parameters
    _, regions = selectivesearch.selective_search(img_np, scale=200, sigma=0.5, min_size=10)
    if len(regions) == 0:
        print(f"Warning: No region proposals generated for image with shape {img_np.shape}.")

    processed_proposals = []

    # Pre-filtering of regions
    for region in regions:
        x, y, w, h = region['rect']
        area = w * h
        if w >= 10 and h >= 10 and 10 <= area <= 0.8 * (img_width * img_height):
            x_max, y_max = x + w, y + h
            processed_proposals.append([x, y, x_max, y_max])

    return processed_proposals

In [12]:
# Function to process a single image
def process_single_image(image_data, image_folder_path):
    img_id = image_data['id']
    img_name = image_data['file_name']
    img_path = os.path.join(image_folder_path, img_name)

    if not os.path.exists(img_path):
        raise ValueError(f"Image not found at path: {img_path}")

    # Load the image using OpenCV (in RGB mode)
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
    original_height, original_width, _ = image.shape

    # Resize the image to speed up Selective Search
    resized_image = cv2.resize(image, (original_width // 2, original_height // 2), interpolation=cv2.INTER_AREA)

    # Generate region proposals on the resized version  
    processed_proposals = generate_and_process_proposals(resized_image, original_width // 2, original_height // 2)

    # Scale the coordinates of the proposals back to the original size
    scaled_proposals = [[x * 2, y * 2, x_max * 2, y_max * 2] for x, y, x_max, y_max in processed_proposals]

    image_data = {
        "image_id": img_id,
        "file_name": img_name,
        "original_size": [original_width, original_height],
        "proposals": []
    }

    for i, proposal in enumerate(scaled_proposals):
        x_min, y_min, x_max, y_max = proposal
        image_data["proposals"].append({
            "proposal_id": i,
            "coordinates": [x_min, y_min, x_max, y_max]
        })

    return image_data

In [13]:
# Function to handle batches
def batch(iterable, n=1):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, n))
        if not chunk:
            break
        yield chunk

In [14]:
def generate_dataset_proposals(coco_json, image_folder_path, output_dir, output_json):
    os.makedirs(output_dir, exist_ok=True)
    all_image_data = []

    # Load the COCO JSON file
    coco_data = load_json(coco_json)

    # Prepare the mapping of annotations for images
    image_annotations_map = {}
    for annotation in coco_data['annotations']:
        image_id = annotation['image_id']
        if image_id not in image_annotations_map:
            image_annotations_map[image_id] = []
        image_annotations_map[image_id].append(annotation)

    # Filter images that contain annotations with category_id == 0 (background)
    images_with_annotations = [
        image_data for image_data in coco_data['images']
        if image_data['id'] in image_annotations_map and len(image_annotations_map[image_data['id']]) > 0
        and not any(annot['category_id'] == 0 for annot in image_annotations_map[image_data['id']])  # Exclude background
    ]

    # Parameters for parallelization and batch processing
    max_workers = os.cpu_count() - 1
    batch_size = 500
    total_batches = len(images_with_annotations) // batch_size + (len(images_with_annotations) % batch_size > 0)

    # Process images in batches with tqdm to monitor batch progress
    with tqdm(total=total_batches, desc="Processing batches") as pbar:
        for image_batch in batch(images_with_annotations, batch_size):
            results = []
            for img in image_batch:
                results.append(process_single_image(img, image_folder_path))
            all_image_data.extend(results)
            pbar.update(1)  # Update the progress bar for each completed batch

    # Save the result in JSON format using orjson
    with open(output_json, 'wb') as json_file:
        json_file.write(orjson.dumps(all_image_data, option=orjson.OPT_INDENT_2))

    print(f"Created JSON file with region proposals: {output_json}")

In [15]:
generate_dataset_proposals(coco_json_path, image_folder_path, proposal_folder_path, output_proposal_json_path)

Processing batches: 100%|██████████| 32/32 [1:59:51<00:00, 224.73s/it]  


Created JSON file with region proposals: ..\data\processed\active_regions_dataset\proposals.json


In [16]:
# Name of the zip file to create
zip_file_name = zip_proposals

# List of files and folders to include in the zip
items_to_zip = [
    coco_json_path, # Include the modified COCO JSON file
    output_proposal_json_path, # Include the proposals JSON file
]

# Function to add files and folders to the zip
def zip_folder(zipf, folder_path, base_folder=""):
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, base_folder)
            zipf.write(file_path, arcname)

# Create the zip
with zipfile.ZipFile(zip_file_name, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for item in items_to_zip:
        if os.path.exists(item):  # Check if the file or folder exists
            if os.path.isdir(item):  # If it's a folder, add all its contents
                zip_folder(zipf, item, PROCESSED_DATA_PATH)
            else:  # If it's a file, add it directly
                zipf.write(item, os.path.relpath(item, PROCESSED_DATA_PATH))
        else:
            print(f"Item not found: {item}")

## Positive region proposals with undersampling

In [17]:
def visualize_image_with_bbox(image_path, bbox, title, ax):
    """
    Visualize an image with a bounding box using Matplotlib in a subplot.
    :param image_path: Path to the image.
    :param bbox: Bounding box in the format [xmin, ymin, xmax, ymax].
    :param title: Title to display above the image.
    :param ax: Axes of the subplot where the image will be drawn.
    """
    # Load the image
    image = Image.open(image_path)
    
    # Extract xmin, ymin, xmax, ymax
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin

    # Add the image and the bounding box
    ax.imshow(image)
    rect = patches.Rectangle(
        (xmin, ymin), width, height,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    
    # Add the title
    ax.set_title(title, fontsize=15)
    ax.axis('off')  # Remove the axes

In [18]:
def calculate_bbox_areas_for_all_categories(json_file_path, images_dir):
    """
    Calculate the minimum, maximum, and average area for each category and draw the corresponding bounding boxes 
    on the images for visualization.

    :param json_file_path: Path to the JSON file containing the annotations.
    :param images_dir: Directory containing the original images.
    """
    # Load the JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # Create dictionaries to map IDs
    areas_per_category = defaultdict(list)
    category_map = {category['id']: category['name'] for category in data['categories']}
    images_map = {image['id']: image for image in data['images']}

    # Collect areas
    for annotation in data.get("annotations", []):
        category_id = annotation["category_id"]
        area = annotation["area"]
        areas_per_category[category_id].append((area, annotation))

    # Iterate over each category
    for category_id, areas in areas_per_category.items():
        if not areas:
            continue

        # Find annotations with minimum, maximum, and calculate the average area
        min_area_annotation = min(areas, key=lambda x: x[0])[1]
        max_area_annotation = max(areas, key=lambda x: x[0])[1]
        avg_area = sum(area for area, _ in areas) / len(areas)

        # Print area information
        print(f"Category: {category_map[category_id]}")
        print(f"  Minimum Area: {min_area_annotation['area']}")
        print(f"  Maximum Area: {max_area_annotation['area']}")
        print(f"  Average Area: {avg_area:.2f}\n")

        # Create a subplot to visualize images with bounding boxes
        fig, ax = plt.subplots(1, 2, figsize=(15, 7))

        # Visualize images with bounding boxes for minimum and maximum area
        for annotation, label, ax_idx in [(min_area_annotation, "Minimum", 0), (max_area_annotation, "Maximum", 1)]:
            image_id = annotation['image_id']
            image_info = images_map.get(image_id)
            if not image_info:
                print(f"Image with ID {image_id} not found.")
                continue

            # Image path
            image_path = os.path.join(images_dir, image_info['file_name'])
            if not os.path.exists(image_path):
                print(f"Image {image_path} not found.")
                continue
            
            # Visualize image with bounding box
            bbox = annotation['bbox']
            title = f"{category_map[category_id]} ({label})"
            visualize_image_with_bbox(image_path, bbox, title, ax[ax_idx])

        plt.show()

In [ ]:
# Example usage
calculate_bbox_areas_for_all_categories(in_new_coco_json_path, image_folder_path)

In [22]:
ignored_count = 0  # Global counter for ignored regions

def get_iou(bb1, bb2):
    global ignored_count  # Access the global counter

    try:
        # Ensure the dimensions are correct
        assert bb1['x1'] < bb1['x2']
        assert bb1['y1'] < bb1['y2']
        assert bb2['x1'] < bb2['x2']
        assert bb2['y1'] < bb2['y2']
    except AssertionError:
        # If an error occurs, increment the counter for ignored regions
        ignored_count += 1
        return 0.0  # Return 0.0 for IoU in case of error (no overlap)

    # Calculate the dimensions of the common area between the two boxes
    x_left = max(bb1['x1'], bb2['x1'])
    y_top = max(bb1['y1'], bb2['y1'])
    x_right = min(bb1['x2'], bb2['x2'])
    y_bottom = min(bb1['y2'], bb2['y2'])

    # If there is no overlap, return 0 as the intersection area
    if x_right < x_left or y_bottom < y_top:
        return 0.0

    # Calculate the intersection area
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    
    # Calculate the individual areas of the two bounding boxes
    bb1_area = (bb1['x2'] - bb1['x1']) * (bb1['y2'] - bb1['y1'])
    bb2_area = (bb2['x2'] - bb2['x1']) * (bb2['y2'] - bb2['y1'])
    
    # Calculate the union area
    iou = intersection_area / float(bb1_area + bb2_area - intersection_area)

    # Ensure the IoU is within the correct range
    assert iou >= 0.0
    assert iou <= 1.0

    return iou

In [23]:
def get_adaptive_threshold(bbox):
    """Calculate an adaptive IoU threshold based on the bounding box area and category."""
    # bbox is a tensor or a list with [x1, y1, x2, y2]
    width = bbox[2] - bbox[0]  # Calculate width
    height = bbox[3] - bbox[1]  # Calculate height
    
    # Calculate the area
    area = width * height

    counter = 0
    
    # Average areas per category (excluding background)
    area_media = {
        2: 651.14,   # Truck
        11: 6075.12  # Aircraft
    }
    
    # Retrieve the minimum and maximum areas for normalization
    min_area = min(area_media.values())
    max_area = max(area_media.values())
    
    # Normalize the area with respect to the defined area range
    normalized_area = (area - min_area) / (max_area - min_area)  # Value between 0 and 1
    
    # Clipping to avoid values out of range
    normalized_area = max(0, min(1, normalized_area))
    
    # Map the normalized value to a threshold range (0.3 to 0.7)
    threshold = 0.3 + 0.4 * normalized_area  # From 0.3 to 0.7
    
    return threshold

In [24]:
def assign_and_save_regions(region_json_path, bbox_json_path, image_dir, output_dir, output_json_path):
    """Assign proposed regions to bounding boxes, save positive regions as images, and create a new JSON with activated information."""
    
    # Load JSON files
    with open(region_json_path, 'r') as f:
        regions = json.load(f)

    with open(bbox_json_path, 'r') as f:
        bboxes = json.load(f)
    
    # Create a dictionary to look up annotations by image_id
    annotations_by_image = {}
    for annot in bboxes["annotations"]:
        img_id = annot["image_id"]
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        bbox = annot["bbox"]
        annotations_by_image[img_id].append((torch.tensor(bbox, dtype=torch.float32), annot["category_id"]))
    
    # Create a dictionary to map category_id to category names
    category_mapping = {cat["id"]: cat["name"] for cat in bboxes["categories"]}
    
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    counter = 0  # Counter for saved images
    
    active_region_data = []  # List for active region data

    # Wrap the main loop for each image with tqdm
    for image in tqdm(regions, desc="Processing images", total=len(regions)):
        image_id = image["image_id"]
        file_name = image["file_name"]
        proposals = image["proposals"]
        
        # Get ground-truth bounding boxes and categories for the current image
        gt_data = annotations_by_image.get(image_id, [])
        if not gt_data:
            # If there are no ground-truth bounding boxes, skip the image
            continue
        
        gt_bboxes = [item[0] for item in gt_data]  # Bounding box ground truth
        gt_categories = [item[1] for item in gt_data]  # Category ground truth
        
        # Transform proposals into a list of dictionaries compatible with get_iou
        proposal_coords = [{'x1': p["coordinates"][0], 'y1': p["coordinates"][1], 
                            'x2': p["coordinates"][2], 'y2': p["coordinates"][3]} 
                           for p in proposals]
        
        # Perform undersampling for category 6 before proceeding
        filtered_proposals = []
        filtered_gt_bboxes = []
        filtered_gt_categories = []
        
        for i, (prop, gt_bbox, gt_category) in enumerate(zip(proposal_coords, gt_bboxes, gt_categories)):
            filtered_proposals.append(prop)
            filtered_gt_bboxes.append(gt_bbox)
            filtered_gt_categories.append(gt_category)

        # Transform back into coordinates for IoU
        proposal_coords = filtered_proposals
        gt_bboxes = filtered_gt_bboxes
        gt_categories = filtered_gt_categories

        # Calculate the IoU matrix using the get_iou function
        iou_matrix = []
        for proposal in proposal_coords:
            iou_row = []
            for gt_bbox in gt_bboxes:
                gt_dict = {'x1': gt_bbox[0].item(), 'y1': gt_bbox[1].item(), 
                           'x2': gt_bbox[2].item(), 'y2': gt_bbox[3].item()}
                iou = get_iou(proposal, gt_dict)
                iou_row.append(iou)
            iou_matrix.append(iou_row)

        # Check if the IoU matrix is empty
        if not iou_matrix:
            continue
        
        iou_matrix = torch.tensor(iou_matrix)

        # Identify positive regions using the adaptive threshold
        positive_indices = []
        zero_iou_indices = []
        for row_idx, row in enumerate(iou_matrix):
            for col_idx, iou in enumerate(row):
                adaptive_threshold = get_adaptive_threshold(gt_bboxes[col_idx])
                if iou >= adaptive_threshold:
                    positive_indices.append((row_idx, col_idx))
                elif iou == 0:
                    # Accumulate images with IoU = 0 for later selection
                    zero_iou_indices.append((row_idx, -1))  # Assign category 0 with -1
        
        # Randomly select 0.5% of images with IoU = 0
        k = int(0.005 * len(zero_iou_indices))
        selected_zero_iou = random.sample(zero_iou_indices, k) if k > 0 else []        
        
        # Add only the selected ones to the positive set
        positive_indices.extend(selected_zero_iou)

        # Load the original image
        image_path = os.path.join(image_dir, file_name)
        original_image = cv2.imread(image_path)
        if original_image is None:
            print(f"Image not found: {image_path}")
            continue

        # Wrap the loop for each positive proposal  
        for row_idx, col_idx in positive_indices:
            if col_idx == -1:
                category_id = 0  
            else:
                category_id = gt_categories[col_idx]
            
            # Add the condition to include only 10% of categories 0 and 6
            if category_id in [0, 6]:
                if random.random() > 0.9:  # Keep 10% of categories 0 and 6
                    # Calculate the bounding box coordinates
                    p = proposal_coords[row_idx]
                    x_min, y_min, x_max, y_max = p["x1"], p["y1"], p["x2"], p["y2"]
                    
                    cropped = original_image[int(y_min):int(y_max), int(x_min):int(x_max)]
                    if cropped.size == 0:
                        continue  # Skip if the cropped region is empty
                        
                    # Resize to 224x224
                    resized = cv2.resize(cropped, (224, 224), interpolation=cv2.INTER_AREA)
                    
                    # Save the image
                    crops_dir = output_dir
                    os.makedirs(crops_dir, exist_ok=True)
                    output_path = crops_dir / f"image_{counter:06d}.jpg"
                    cv2.imwrite(str(output_path), resized)
                    
                    # Add the activated proposal to the new JSON in COCO format
                    active_region_data.append({
                        "image_id": image_id,
                        "file_name": file_name,
                        "category_id": category_id,
                        "proposal_id": row_idx,
                        "region_bbox": [x_min, y_min, x_max, y_max],  # Use xmin, ymin, xmax, ymax
                        "original_bbox": gt_bboxes[col_idx].tolist() if col_idx != -1 else [],  # Add the original bbox
                        "saved_path": str(output_path)
                    })
                    
                    counter += 1
            else:
                # If the category is not 0 or 6, always include the proposal
                # Calculate the bounding box coordinates
                p = proposal_coords[row_idx]
                x_min, y_min, x_max, y_max = p["x1"], p["y1"], p["x2"], p["y2"]
                
                cropped = original_image[int(y_min):int(y_max), int(x_min):int(x_max)]
                if cropped.size == 0:
                    continue  # Skip if the cropped region is empty

                # Resize to 224x224
                resized = cv2.resize(cropped, (224, 224), interpolation=cv2.INTER_AREA)
                
                # Save the image
                crops_dir = output_dir
                os.makedirs(crops_dir, exist_ok=True)

                output_path = crops_dir / f"image_{counter:06d}.jpg"
                cv2.imwrite(str(output_path), resized)
                
                # Add the activated proposal to the new JSON in COCO format
                active_region_data.append({
                    "image_id": image_id,
                    "file_name": file_name,
                    "category_id": category_id,
                    "proposal_id": row_idx,
                    "region_bbox": [x_min, y_min, x_max, y_max],  # Use xmin, ymin, xmax, ymax
                    "original_bbox": gt_bboxes[col_idx].tolist(),  # Add the original bbox
                    "saved_path": str(output_path)
                })
                
                counter += 1

    # Save the new JSON with the active regions
    with open(output_json_path, 'w') as json_file:
        json.dump(active_region_data, json_file, indent=2)

    print(counter)

In [25]:
# Execute the assignment and obtain IoU values

assign_and_save_regions(proposal_json_path, in_new_coco_json_path, image_folder_path, active_region_folder_path / "proposals", actproposal_json_path)

Processing images: 100%|██████████| 15517/15517 [54:23<00:00,  4.76it/s]  


14275


In [26]:
def analyze_regions(file_path):
    """
    Analyze a JSON file to obtain the number of regions and the occurrences of category_id.

    :param file_path: Path to the JSON file containing the annotations.
    :return: Tuple containing the number of regions and a dictionary with the occurrences of category_id.
    """
    # Load the JSON file
    data = load_json(file_path)

    # Count the number of regions
    num_regions = len(data)
    
    # Get the occurrences of category_id
    category_ids = [entry['category_id'] for entry in data]
    category_counts = Counter(category_ids)

    # Sort the occurrences by category ID
    sorted_category_counts = dict(sorted(category_counts.items()))

    return num_regions, sorted_category_counts

In [27]:
num_regions, category_counts = analyze_regions(actproposal_json_path)
print(f"Number of regions: {num_regions}")
print("Occurrences of category_id:", category_counts)

Number of regions: 14275
Occurrences of category_id: {0: 3510, 1: 2372, 2: 1764, 3: 275, 4: 574, 5: 288, 6: 4883, 7: 6, 8: 88, 9: 415, 10: 21, 11: 79}


In [28]:
# List of files and folders to include in the zip
items_to_zip = [
    in_new_coco_json_path,
    output_proposal_json_path.parent / "proposals"
]

# Function to add files and folders to the zip
def zip_folder(zipf, folder_path, base_folder=""):
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, base_folder)
            zipf.write(file_path, arcname)

# Create the zip
with zipfile.ZipFile(zip_active_regions, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for item in items_to_zip:
        if os.path.exists(item):  # Check if the file or folder exists
            if os.path.isdir(item):  # If it's a folder, add all its contents
                zip_folder(zipf, item, active_region_folder_path)
            else:  # If it's a file, add it directly
                zipf.write(item, os.path.relpath(item, PROCESSED_DATA_PATH))
        else:
            print(f"Item not found: {item}")

## Category distribution

In [ ]:
data = load_json(actproposal_json_path)
 
# Count the occurrences of each category
category_counts = Counter(item['category_id'] for item in data)
 
# Prepare the data for the plot
categories = list(category_counts.keys())
counts = list(category_counts.values())
 
# Create the plot
plt.figure(figsize=(10, 6))
plt.bar(categories, counts, color='skyblue')
plt.xlabel('Category ID')
plt.ylabel('Frequency')
plt.title('Distribution of categories in the dataset')
plt.xticks(categories)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Splitting

In [ ]:
print(data["category_id"].value_counts())

category_id
6     4883
0     3510
1     2372
2     1764
4      574
9      415
5      288
3      275
8       88
11      79
10      21
7        6
Name: count, dtype: int64


because there are only 6 helipads (class 7), it is not possible to have only 1 in val, 1 in test and 4 in train, so I will include helipads in the building class (class 6) so that splitting works

In [35]:
# Load the dataset from JSON

data = load_json(actproposal_json_path)

# Convert to DataFrame for easier handling
df = pd.DataFrame(data)

# Create a temporary column for stratification
df["category_id_strat"] = df["category_id"].replace({7: 6})  # Helipad → Building

# Extract the data and labels
X = df.index  # Row indices
y = df["category_id_strat"]  # Label for stratification

# Step 1: Train + Val/Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 2: Val/Test split
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Create the final datasets
train_data = df.loc[X_train].drop(columns=["category_id_strat"])
val_data = df.loc[X_val].drop(columns=["category_id_strat"])
test_data = df.loc[X_test].drop(columns=["category_id_strat"])

# Save the datasets to new JSON files
train_data.to_json(train_path, orient="records")
val_data.to_json(val_path, orient="records")
test_data.to_json(test_path, orient="records")

print("Splitting completed. Files saved: train.json, val.json, test.json.")

# Percentage of data for each set
total_data = len(df)
print("\nPercentage of data for each set:")
print(f"Train: {len(train_data) / total_data:.2%}")
print(f"Validation: {len(val_data) / total_data:.2%}")
print(f"Test: {len(test_data) / total_data:.2%}")

# Class distribution for each set
print("\nClass distribution:")
train_class_dist = train_data["category_id"].value_counts(normalize=True) * 100
val_class_dist = val_data["category_id"].value_counts(normalize=True) * 100
test_class_dist = test_data["category_id"].value_counts(normalize=True) * 100

print("Train set:")
print(train_class_dist.sort_index())

print("\nValidation set:")
print(val_class_dist.sort_index())

print("\nTest set:")
print(test_class_dist.sort_index())

Splitting completed. Files saved: train.json, val.json, test.json.

Percentage of data for each set:
Train: 80.00%
Validation: 10.00%
Test: 10.00%

Class distribution:
Train set:
category_id
0     24.588441
1     16.619965
2     12.355517
3      1.926445
4      4.019264
5      2.014011
6     34.220665
7      0.026270
8      0.621716
9      2.907180
10     0.148862
11     0.551664
Name: proportion, dtype: float64

Validation set:
category_id
0     24.597057
1     16.608269
2     12.333567
3      1.962158
4      3.994394
5      2.032235
6     34.197617
7      0.070077
8      0.630694
9      2.873160
10     0.140154
11     0.560617
Name: proportion, dtype: float64

Test set:
category_id
0     24.579832
1     16.596639
2     12.394958
3      1.890756
4      4.061625
5      2.030812
6     34.103641
7      0.140056
8      0.560224
9      2.941176
10     0.140056
11     0.560224
Name: proportion, dtype: float64


In [ ]:
train_data = load_json(train_path)

# Visualize 3 random images from the train set
fig, axes = plt.subplots(1, 3, figsize=(20, 10))
for i, ax in enumerate(axes):
    sample_entry = random.choice(train_data)
    image_path = sample_entry["saved_path"]
    title = f"Category ID: {sample_entry['category_id']}"
    visualize_image_with_bbox(image_path, [0,0,0,0], title, ax)

plt.tight_layout()
plt.show()

val_data = load_json(val_path)

# Visualize 3 random images from the validation set
fig, axes = plt.subplots(1, 3, figsize=(20, 10))
for i, ax in enumerate(axes):
    sample_entry = random.choice(val_data)
    image_path = sample_entry["saved_path"]
    title = f"Category ID: {sample_entry['category_id']}"
    visualize_image_with_bbox(image_path, [0,0,0,0], title, ax)

plt.tight_layout()
plt.show()

test_data = load_json(test_path)

# Visualize 3 random images from the test set
fig, axes = plt.subplots(1, 3, figsize=(20, 10))
for i, ax in enumerate(axes):
    sample_entry = random.choice(test_data)
    image_path = sample_entry["saved_path"]
    title = f"Category ID: {sample_entry['category_id']}"
    visualize_image_with_bbox(image_path, [0,0,0,0], title, ax)

plt.tight_layout()
plt.show()

## Custom dataset

In [67]:
class CustomDataset(Dataset):
    def __init__(self, json_file, transform=None):
        """
        Initialize the dataset.

        :param json_file: Path to the JSON file containing region information.
        :param transform: Transformations to apply to the images. If not provided, default transformations are used.
        """
        # Load the JSON file
        self.data = load_json(json_file)
        
        # Default transformations if none are provided
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),         # Resize the image to 224x224
            transforms.ToTensor(),                  # Convert the image to a tensor
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalization for pre-trained models
        ])  

    def __len__(self):
        """Returns the total number of images/proposals in the dataset."""
        return len(self.data)

    def __getitem__(self, idx):
        """Returns a single example (image and label) for training."""
        # Load the example from the JSON file
        sample = self.data[idx]
        
        # Load the image
        image = Image.open(sample["saved_path"]).convert("RGB")
        
        # Category label
        label = sample["category_id"]  # Proposal category

        # Apply transformations
        image = self.transform(image)
        
        return image, label

In [68]:
test_ds = CustomDataset(test_path)
train_ds = CustomDataset(train_path)
val_ds = CustomDataset(val_path)

TrainLoader = DataLoader(train_ds, batch_size=32, shuffle=True)
ValLoader = DataLoader(val_ds, batch_size=32, shuffle=False)
TestLoader = DataLoader(test_ds, batch_size=32, shuffle=False)

## Classification

In [69]:
def plot_loss(train_losses, val_losses):
    """
    Function to plot the loss function during training and validation.

    :param train_losses: List of losses during training.
    :param val_losses: List of losses during validation.
    """
    epochs = range(1, len(train_losses) + 1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label="Train Loss", color="blue", linestyle='-', marker='o')
    plt.plot(epochs, val_losses, label="Validation Loss", color="red", linestyle='-', marker='x')
    
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [70]:
def train_model(net, train_loader, val_loader, criterion, optimizer, device, epochs, path_min_loss, num_classes, clip_value=1.0):
    min_val_loss = float('inf')
    
    # Lists to record losses during training and validation
    train_losses = []
    val_losses = []
    
    # Dictionary to save features
    train_features = {}  # Key: epoch, Value: list of features

    # Add a learning rate scheduler to reduce the learning rate when the validation loss does not improve
    scheduler = ReduceLROnPlateau(optimizer, 'min', patience=5) # verbose = True

    for epoch in range(epochs):
        net.train()  # Training mode
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        epoch_features = []  # To record features for each batch
        class_probs = torch.zeros((len(train_loader.dataset), num_classes)).to(device)  # Class probabilities

        # Progress bar for training
        train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training", leave=False)

        for i, (images, labels) in enumerate(train_progress):
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            features, outputs = net(images)

            # Save features for the current batch
            epoch_features.append(features.detach().cpu().numpy())

            # Calculate the weighted loss
            loss = criterion(outputs, labels)

            # Predict the class with the highest probability
            _, predicted = outputs.max(1)

            # Calculate the probabilities for each class (softmax)
            probs = torch.softmax(outputs, dim=1)
            class_probs[i * images.size(0):(i + 1) * images.size(0)] = probs

            # Monitor gradients
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            utils.clip_grad_norm_(net.parameters(), clip_value)

            optimizer.step()

            # Statistics
            train_loss += loss.item() * images.size(0)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

            # Update the progress bar with the current loss
            train_progress.set_postfix(loss=loss.item(), accuracy=100. * correct_train / total_train)

        # Save features for the current epoch
        train_features[epoch] = epoch_features

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_accuracy = 100. * correct_train / total_train
        train_losses.append(avg_train_loss)

        # Calculate the average probability for each class
        avg_class_probs = class_probs.mean(dim=0)
        print(f"Average class probabilities in epoch {epoch+1}: {avg_class_probs}")

        # Validation
        net.eval()  # Validation mode
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        # Progress bar for validation
        val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Validation", leave=False)

        with torch.no_grad():
            for images, labels in val_progress:
                images, labels = images.to(device), labels.to(device)

                # Forward pass
                _, outputs = net(images)
                
                # Predict the class with the highest probability
                _, predicted = outputs.max(1)

                # Calculate the loss
                loss = criterion(outputs, labels)

                # Statistics
                val_loss += loss.item() * images.size(0)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()

                # Update the progress bar with the loss and accuracy
                val_progress.set_postfix(loss=loss.item(), accuracy=100. * correct_val / total_val)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100. * correct_val / total_val
        val_losses.append(avg_val_loss)

        # Save the model with the lowest validation loss
        if avg_val_loss < min_val_loss:
            print(f"Saving the best model: Val Loss improved from {min_val_loss:.4f} to {avg_val_loss:.4f}")
            min_val_loss = avg_val_loss
            torch.save(net.state_dict(), path_min_loss)

        # Update the learning rate based on the validation loss
        scheduler.step(avg_val_loss)

        # Print statistics for the epoch
        print(f"Epoch [{epoch + 1}/{epochs}]")
        print(f"Train Loss: {avg_train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")
        print(f"Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

    print("Training completed!")

    # Call the function to plot the graph
    plot_loss(train_losses, val_losses)

    return train_features

In [71]:
def test_model(net, test_loader, criterion, device, path_min_loss, json_path, original_img_path, reference_json_path):
    """
    Function to test the model and visualize the results on original images.

    :param net: Model to be tested.
    :param test_loader: DataLoader for the test set.
    :param criterion: Loss function.
    :param device: Device (CPU or GPU).
    :param path_min_loss: Path to the saved model.
    :param json_path: Path to the JSON file containing test set image details.
    :param original_img_path: Path to the folder containing original images.
    :param reference_json_path: Path to the JSON file containing original image information.
    """
    # Load the best saved model (with the weights_only=True parameter to avoid the warning)
    net.load_state_dict(torch.load(path_min_loss, map_location=device))
    net.eval()

    test_loss = 0.0
    correct_test = 0
    total_test = 0

    all_labels = []
    all_predictions = []

    # Load JSON files
    with open(json_path, 'r') as f:
        test_data = json.load(f)

    with open(reference_json_path, 'r') as f:
        reference_data = json.load(f)

    # Create a dictionary for quick lookup of original images
    id_to_filename = {img['id']: img['file_name'] for img in reference_data['images']}

    # Create a dictionary to group bounding boxes by image_id
    image_id_to_bboxes = {}
    for image_info in test_data:
        image_id = image_info['image_id']
        region_bbox = image_info['region_bbox']
        if image_id not in image_id_to_bboxes:
            image_id_to_bboxes[image_id] = []
        image_id_to_bboxes[image_id].append(region_bbox)

    with torch.no_grad():
        for idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            _, outputs = net(images)

            # Calculate the loss
            loss = criterion(outputs, labels)

            # Statistics
            test_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_test += labels.size(0)
            correct_test += predicted.eq(labels).sum().item()

            # Save all labels and predictions
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

            # Show some examples
            if idx < 5:  # Show the first 5 batches
                for i in range(min(len(images), 3)):  # Show up to 3 images per batch
                    image_info = test_data[idx * len(images) + i]  # Retrieve image info from JSON
                    image_id = image_info['image_id']

                    # Find the file_name using the image_id
                    file_name = id_to_filename.get(image_id)
                    if not file_name:
                        print(f"Image with ID {image_id} not found in the reference JSON.")
                        continue

                    # Path to the original image
                    img_path = os.path.join(original_img_path, file_name)

                    # Load the original image
                    img = cv2.imread(img_path)
                    if img is None:
                        print(f"Image not found: {img_path}")
                        continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                    # Draw bounding boxes for the real labels (blue)
                    bboxes_real = image_id_to_bboxes.get(image_id, [])
                    for bbox in bboxes_real:
                        xmin, ymin, xmax, ymax = bbox
                        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 0, 255), 2)  # Blue for real

                    # Show the image with bounding boxes (without labels written on the image)
                    plt.figure(figsize=(6, 6))
                    plt.imshow(img)
                    plt.axis('off')
                    plt.show()

                    # Now print the predicted and real labels below the image
                    print(f"Predictions vs Reality for image {file_name}:")
                    for j, bbox in enumerate(bboxes_real):
                        pred_label = predicted[i].item()
                        # Print the predicted and real values
                        print(f"  Bounding Box {j + 1}: Predicted: {pred_label}, Actual: {labels[i].item()}")

    avg_test_loss = test_loss / len(test_loader.dataset)
    test_accuracy = 100. * correct_test / total_test

    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

    # Calculate and visualize the confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize by row

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=True, cmap='Blues', fmt='.2f', xticklabels=np.unique(all_labels), yticklabels=np.unique(all_labels))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

In [72]:
# We use a pretrained AlexNet

from torchvision import models
from torchvision.models import AlexNet_Weights

num_classes = 12

# Load pretrained AlexNet on ImageNet
alexnet = models.alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)

# Adapt the final fully connected layer to match the number of classes in our dataset
in_features = alexnet.classifier[6].in_features
alexnet.classifier[6] = nn.Linear(in_features, num_classes)

# Wrapper to keep (features, output)
class AlexNetFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features_part = model.features
        self.avgpool = model.avgpool
        self.classifier_part = model.classifier

    def forward(self, x):
        x = self.features_part(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # Jusqu’à l’avant-dernière couche
        x = self.classifier_part[:-1](x)
        features = x

        outputs = self.classifier_part[-1](x)
        return features, outputs

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AlexNetFeatureExtractor(alexnet).to(device)

criterion = nn.CrossEntropyLoss().to(device)

learning_rate = 0.0001   # plus petit pour finetuning
epochs = 10              
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_features = train_model(
    net=model,
    train_loader=TrainLoader,
    val_loader=ValLoader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    path_min_loss=alexnet_path,
    num_classes = num_classes
)

In [ ]:
test_model(
    net=model,
    test_loader=TestLoader,
    criterion=criterion,
    device=device,
    path_min_loss=alexnet_path,
    json_path = test_path,
    original_img_path = image_folder_path,
    reference_json_path = in_new_coco_json_path
)

In [ ]:
# Next we use a pretrained ResNet50

from torchvision.models import ResNet50_Weights

# Number of classes
num_classes = 12

# Pre-trained ResNet50 model
resnet50 = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Modify the last fully connected layer for the number of classes
in_features = resnet50.fc.in_features  # Extract the input features of the last layer
resnet50.fc = nn.Linear(in_features, num_classes)  # New classification layer

# Move the model to GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet50 = resnet50.to(device)

# Modify the forward to return both features and final predictions
class ResNet50FeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = nn.Sequential(*list(model.children())[:-1])
        self.fc = model.fc

    def forward(self, x):
        x = self.features(x)
        features = torch.flatten(x, 1)
        outputs = self.fc(features)
        return features, outputs

# Create the new model with feature extraction
model = ResNet50FeatureExtractor(resnet50).to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00005)

# Number of epochs
epochs = 15

# Training function
train_features = train_model(
    net=model,
    train_loader=TrainLoader,
    val_loader=ValLoader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    path_min_loss=resnet_path,
    num_classes=num_classes
)

In [ ]:
test_model(
    net=model,
    test_loader=TestLoader,
    criterion=criterion,
    device=device,
    path_min_loss=resnet_path,
    json_path = test_path,
    original_img_path = image_folder_path,
    reference_json_path = in_new_coco_json_path
)

## Data box regressor

In [77]:
def test_model_regressor(net, test_loader, criterion, device, path_min_loss, pred_json_path):
    """
    Function to test the model, replace the category_id with the predicted one, and visualize the results on original images.

    :param net: Model to be tested.
    :param test_loader: DataLoader for the test set.
    :param criterion: Loss function.
    :param device: Device (CPU or GPU).
    :param path_min_loss: Path to the saved model.
    :param json_path: Path to the JSON file containing test set image details.
    :param pred_json_path: Path to the JSON file to be modified with predicted category_id.
    """
    # Load the best saved model
    net.load_state_dict(torch.load(path_min_loss, map_location=device))
    net.eval()

    all_labels = []
    all_predictions = []

    # Load the JSON with image information to be modified
    with open(pred_json_path, 'r') as f:
        pred_data = json.load(f)

    with torch.no_grad():
        for idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            _, outputs = net(images)

            # Calculate the loss
            loss = criterion(outputs, labels)

            # Statistics
            _, predicted = outputs.max(1)

            # Save all labels and predictions
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

            # Modify the JSON with predictions
            for i in range(len(images)):
                image_info = pred_data[idx * len(images) + i]  # Retrieve image info from JSON
                image_id = image_info['image_id']

                # Find the entry in pred_json and replace the category_id with the predicted one
                for entry in pred_data:
                    if entry['image_id'] == image_id:
                        entry['category_id'] = int(predicted[i].item())

    # Save the modified JSON
    with open(pred_json_path, 'w') as f:
        json.dump(pred_data, f, indent=4)

    # Calculate the test accuracy
    test_accuracy = 100. * sum(np.array(all_labels) == np.array(all_predictions)) / len(all_labels)
    print(f"Test Accuracy: {test_accuracy:.2f}%")

    # Calculate and visualize the confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize by row

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=True, cmap='Blues', fmt='.2f', xticklabels=np.unique(all_labels), yticklabels=np.unique(all_labels))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
# Load JSON data for validation and test
val_json = load_json(val_path)
test_json = load_json(test_path)
reg_json = val_json + test_json

# Save the merged JSON to a new file
with open(regressor_path, 'w') as f_merged:
    json.dump(reg_json, f_merged, indent=4)
    
reg_ds = CustomDataset(regressor_path)
RegLoader = DataLoader(reg_ds, batch_size=32, shuffle=False)

# Call the test function
test_model_regressor(
    net=model,
    test_loader=RegLoader,
    criterion=criterion,
    device=device,
    path_min_loss=resnet_path,
    pred_json_path=regressor_path
)

## Box regressor

In [ ]:
class BoundingBoxRegressor:
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.metrics = {}  # To store performance metrics

    def parse_json(self, json_data):
        proposals = []
        ground_truths = []
        categories = []
        file_names = []
        filtered_data = []

        for item in json_data:
            region_bbox = item.get("region_bbox", [])
            gt_bbox = item.get("original_bbox", [])
            category_id = item.get("category_id", 0)
            file_name = item.get("file_name", "")

            if len(region_bbox) != 4:
                continue

            if category_id == 0 or not gt_bbox:
                gt_bbox = region_bbox

            if len(gt_bbox) != 4:
                continue

            proposals.append(region_bbox)
            ground_truths.append(gt_bbox)
            categories.append(category_id)
            file_names.append(file_name)
            filtered_data.append(item)

        return (
            np.array(proposals, dtype=float),
            np.array(ground_truths, dtype=float),
            np.array(categories, dtype=int),
            file_names,
            filtered_data,
        )

    def train(self, json_data):
        proposals, ground_truths, categories, _, _ = self.parse_json(json_data)
        unique_categories = np.unique(categories)

        min_samples = 10
        
        for category in tqdm(unique_categories, desc="Training Categories"):
            mask = categories == category
            cat_proposals = proposals[mask]
            cat_ground_truths = ground_truths[mask]

            if len(cat_proposals) < min_samples:
                print(f"Skipping category {category}: {len(cat_proposals)} samples")
                continue

            # Calculation of NORMALIZED offsets
            widths = cat_proposals[:, 2] - cat_proposals[:, 0]
            heights = cat_proposals[:, 3] - cat_proposals[:, 1]
            
            offsets = cat_ground_truths - cat_proposals
            offsets[:, [0, 2]] /= widths[:, None]  # Normalize dx
            offsets[:, [1, 3]] /= heights[:, None]  # Normalize dy

            # Normalization of features
            scaler = StandardScaler()
            normalized_proposals = scaler.fit_transform(cat_proposals)

            # Training with validation
            from sklearn.model_selection import train_test_split
            from sklearn.metrics import mean_squared_error
            
            X_train, X_val, y_train, y_val = train_test_split(
                normalized_proposals, offsets, test_size=0.2, random_state=42
            )
            
            model = LinearRegression()
            model.fit(X_train, y_train)
            
            # Performance metric
            val_pred = model.predict(X_val)
            mse = mean_squared_error(y_val, val_pred)
            
            self.models[category] = model
            self.scalers[category] = scaler
            self.metrics[category] = mse
            
            print(f"Category {category}: MSE = {mse:.4f}")

        print(f"Trained {len(self.models)} models")

    def predict(self, category_id, region_bbox, image_width=None, image_height=None):
        if category_id not in self.models:
            print(f"Warning: No model for category {category_id}")
            return region_bbox

        region_bbox = np.array(region_bbox, dtype=float).reshape(1, -1)

        # Normalization
        scaler = self.scalers[category_id]
        normalized_bbox = scaler.transform(region_bbox)

        # Prediction of normalized offsets
        model = self.models[category_id]
        predicted_offsets = model.predict(normalized_bbox)

        # Denormalization of offsets
        width = region_bbox[0][2] - region_bbox[0][0]
        height = region_bbox[0][3] - region_bbox[0][1]
        
        predicted_offsets[0][[0, 2]] *= width
        predicted_offsets[0][[1, 3]] *= height

        # Application
        refined_bbox = (region_bbox + predicted_offsets).flatten()

        # Validation
        if refined_bbox[0] >= refined_bbox[2] or refined_bbox[1] >= refined_bbox[3]:
            print(f"Warning: Invalid bbox, returning original")
            return region_bbox.flatten().tolist()

        # Clip to image dimensions
        if image_width and image_height:
            refined_bbox[0] = max(0, min(refined_bbox[0], image_width))
            refined_bbox[1] = max(0, min(refined_bbox[1], image_height))
            refined_bbox[2] = max(0, min(refined_bbox[2], image_width))
            refined_bbox[3] = max(0, min(refined_bbox[3], image_height))

        return refined_bbox.tolist()

    def save_models(self, output_path):
        with open(output_path, 'wb') as f:
            pickle.dump({
                'models': self.models,
                'scalers': self.scalers,
                'metrics': self.metrics
            }, f)
        print(f"Models saved to {output_path}")

    def load_models(self, model_path):
        with open(model_path, 'rb') as f:
            data = pickle.load(f)
            self.models = data['models']
            self.scalers = data['scalers']
            self.metrics = data.get('metrics', {})
        print(f"Models loaded from {model_path}")
    
    def compute_iou(self, boxA, boxB):
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])

        interW = max(0, xB - xA)
        interH = max(0, yB - yA)
        interArea = interW * interH

        boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

        union = boxAArea + boxBArea - interArea + 1e-6

        return interArea / union


    def evaluate(self, json_data):
        proposals, ground_truths, categories, _, _ = self.parse_json(json_data)

        results = {}

        for category in np.unique(categories):
            if category not in self.models:
                continue

            mask = categories == category
            cat_proposals = proposals[mask]
            cat_gt = ground_truths[mask]

            ious = []
            maes = []

            for proposal, gt in zip(cat_proposals, cat_gt):
                pred = self.predict(category, proposal)

                # IoU
                iou = self.compute_iou(pred, gt)
                ious.append(iou)

                # MAE bbox coords
                mae = mean_absolute_error(gt, pred)
                maes.append(mae)

            if len(ious) > 0:
                results[category] = {
                    "mean_iou": float(np.mean(ious)),
                    "mae": float(np.mean(maes)),
                    "samples": len(ious)
                }

        return results
    
    def predict_json(self, input_json_path, output_json_path):

        with open(input_json_path, "r") as f:
            data = json.load(f)

        refined_data = []

        for item in tqdm(data, desc="Refining BBoxes"):
            region_bbox = item.get("region_bbox", [])
            category_id = item.get("category_id", 0)

            if len(region_bbox) != 4:
                refined_data.append(item)
                continue

            refined_bbox = self.predict(category_id, region_bbox)

            item["refined_bbox"] = refined_bbox
            refined_data.append(item)

        with open(output_json_path, "w") as f:
            json.dump(refined_data, f, indent=4)

        print(f"Refined JSON saved to {output_json_path}")

In [103]:
# Training
regressor = BoundingBoxRegressor()
train_json = load_json(train_path)
regressor.train(train_json)

Training Categories: 100%|██████████| 12/12 [00:00<00:00, 136.74it/s]

Category 0: MSE = 0.0000
Category 1: MSE = 0.0442
Category 2: MSE = 0.0485
Category 3: MSE = 0.0482
Category 4: MSE = 0.0426
Category 5: MSE = 0.0397
Category 6: MSE = 0.0468
Skipping category 7: 3 samples
Category 8: MSE = 0.0808
Category 9: MSE = 0.0491
Category 10: MSE = 0.0299
Category 11: MSE = 0.0386
Trained 11 models


In [ ]:
# Predictions
val_json = load_json(val_path)
metrics = regressor.evaluate(val_json)
print("Validation Metrics:")
for cat, met in metrics.items():
    print(f"Category {cat}: IoU={met['mean_iou']:.3f}, MAE={met['mae']:.3f}")

# Save
regressor.save_models(regressor_path)

# Prediction on new data
regressor.predict_json(test_path, regressor_output_path)

Validation Metrics:
Category 0: IoU=1.000, MAE=0.000
Category 1: IoU=0.474, MAE=6.024
Category 2: IoU=0.492, MAE=5.735
Category 3: IoU=0.547, MAE=6.978
Category 4: IoU=0.523, MAE=6.181
Category 5: IoU=0.492, MAE=6.895
Category 6: IoU=0.568, MAE=6.501
Category 8: IoU=0.490, MAE=6.933
Category 9: IoU=0.548, MAE=7.347
Category 10: IoU=0.534, MAE=8.692
Category 11: IoU=0.555, MAE=9.716
Models saved to ..\models\ResNet50\regressor_best.pth


Refining BBoxes:  62%|██████▏   | 880/1428 [00:00<00:00, 1598.44it/s]

Refining BBoxes: 100%|██████████| 1428/1428 [00:00<00:00, 1716.04it/s]


Refined JSON saved to ..\models\ResNet50\regressed_boxes.json


In [ ]:
def plot_region_boxes(image_path, regions1, regions2, labels1, labels2, image_title, title1, title2):
    """
    Visualize an image with two sets of region boxes and their corresponding labels side by side.

    Parameters:
        image_path (str): Path to the image.
        regions1 (list): List of regions for the first image ([x_min, y_min, x_max, y_max]).
        regions2 (list): List of regions for the second image ([x_min, y_min, x_max, y_max]).
        labels1 (list): List of category_id for the first image.
        labels2 (list): List of category_id for the second image.
        image_title (str): General title for the image pair (image name).
        title1 (str): Subtitle for the first image.
        title2 (str): Subtitle for the second image.
    """
    # Open the image
    img = Image.open(image_path)
    
    # Create a figure with two side-by-side subplots
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(image_title, fontsize=16)  # General title for the pair
    
    # Display the first image with boxes and labels
    axes[0].imshow(img)
    for region, label in zip(regions1, labels1):
        x_min, y_min, x_max, y_max = region
        width = x_max - x_min
        height = y_max - y_min
        rect = plt.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor='r', facecolor='none')
        axes[0].add_patch(rect)
        axes[0].text(x_min, y_min - 5, str(label), color='r', fontsize=8, ha='left', backgroundcolor='white')
    axes[0].set_title(title1, fontsize=12)
    axes[0].axis('off')
    
    # Display the second image with boxes and labels
    axes[1].imshow(img)
    for region, label in zip(regions2, labels2):
        x_min, y_min, x_max, y_max = region
        width = x_max - x_min
        height = y_max - y_min
        rect = plt.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor='r', facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x_min, y_min - 5, str(label), color='r', fontsize=8, ha='left', backgroundcolor='white')
    axes[1].set_title(title2, fontsize=12)
    axes[1].axis('off')
    
    # Display the figure
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout leaving space for the general title
    plt.show()

def visualize_boxes(reg_file, new_reg_file, inactproposals_file, img_folder, selected_files=None):
    """
    Visualize images side by side with overlaid region boxes and corresponding labels from two JSON files.

    Parameters:
        reg_file (str): Path to the reg.json file.
        new_reg_file (str): Path to the new_reg.json file.
        inactproposals_file (str): Path to the inactproposals_json file.
        img_folder (str): Path to the folder containing the images.
        selected_files (list, optional): List of image file names to visualize. If None, use all images.
    """
    # Load data from JSON files
    with open(reg_file, 'r') as f:
        reg_data = json.load(f)

    with open(new_reg_file, 'r') as f:
        new_reg_data = json.load(f)

    with open(inactproposals_file, 'r') as f:
        inactproposals_data = json.load(f)

    # Map file_name to image_id using inactproposals_json
    file_to_id = {item['file_name']: item['image_id'] for item in inactproposals_data}

    # Filter images if `selected_files` is specified
    files_to_visualize = selected_files if selected_files else file_to_id.keys()

    # Iterate over the selected images
    for file_name in files_to_visualize:
        if file_name not in file_to_id:
            print(f"File {file_name} not found in inactproposals_json.")
            continue

        image_id = file_to_id[file_name]

        # Find all region boxes and category_ids associated with this image_id in the two JSON files
        reg_bboxes = []
        reg_labels = []
        for item in reg_data:
            if item['image_id'] == image_id:
                reg_bboxes.append(item['region_bbox'])
                reg_labels.append(item['category_id'])  # Use category_id as label

        new_reg_bboxes = []
        new_reg_labels = []
        for item in new_reg_data:
            if item['image_id'] == image_id:
                new_reg_bboxes.append(item['region_bbox'])
                new_reg_labels.append(item['category_id'])  # Use category_id as label

        # Build the full image path
        image_path = os.path.join(img_folder, file_name)

        if not os.path.exists(image_path):
            print(f"Image {image_path} does not exist.")
            continue

        # Visualize the image with side-by-side boxes and category_ids
        plot_region_boxes(
            image_path, 
            reg_bboxes, 
            new_reg_bboxes, 
            reg_labels, 
            new_reg_labels, 
            f"Image: {file_name}",  # General title
            "Boxes before Box Regressor",  # Subtitle image 1
            "Boxes after Box Regressor"    # Subtitle image 2   
        )

In [ ]:
visualize_boxes(regressor_input_path, regressor_output_path, actproposal_json_path, image_folder_path, selected_files=["img_1896_0_2560.jpg", "img_322_1280_2240.jpg"])

In [107]:
def find_top_images(json_data, top_n=4):
    # A dictionary to count the number of bboxes for each image_id per category
    category_image_bbox_count = defaultdict(lambda: defaultdict(int))

    # Iterate through the JSON data and count the bboxes for each image_id and category_id
    for entry in json_data:
        image_id = entry['image_id']
        category_id = entry['category_id']  # Assume the category is in 'category_id'
        category_image_bbox_count[category_id][image_id] += 1

    # A dictionary to store the final results
    top_images_by_category = {}

    # For each category, sort the images by the number of bboxes (in descending order)
    for category_id, image_count in category_image_bbox_count.items():
        sorted_images = sorted(image_count.items(), key=lambda x: x[1], reverse=True)
        top_images_by_category[category_id] = sorted_images[:top_n]

    return top_images_by_category

# Load the JSON file
with open(regressor_input_path, 'r') as f:
    data = json.load(f)

# Call the function to find the images with the most regions
top_images = find_top_images(data)

# Print the results for each category
if top_images:
    for category_id, top_images_list in top_images.items():
        print(f"\nCategory ID: {category_id}")
        for rank, (image_id, bbox_count) in enumerate(top_images_list, start=1):
            print(f"{rank}. Image ID: {image_id}, Number of bbox: {bbox_count}")
else:
    print("No images found.")


Category ID: 0
1. Image ID: 41822402880, Number of bbox: 105
2. Image ID: 21728801920, Number of bbox: 34
3. Image ID: 11276402880, Number of bbox: 32
4. Image ID: 21725602560, Number of bbox: 29

Category ID: 6
1. Image ID: 1361960960, Number of bbox: 15
2. Image ID: 1831320640, Number of bbox: 10
3. Image ID: 136112800, Number of bbox: 10
4. Image ID: 5489602560, Number of bbox: 9

Category ID: 2
1. Image ID: 12129601280, Number of bbox: 33
2. Image ID: 3796402880, Number of bbox: 30
3. Image ID: 240019203200, Number of bbox: 28
4. Image ID: 11752560640, Number of bbox: 28

Category ID: 11
1. Image ID: 238612801600, Number of bbox: 7
2. Image ID: 14783201280, Number of bbox: 6
3. Image ID: 11143201920, Number of bbox: 5
4. Image ID: 16921920320, Number of bbox: 4

Category ID: 1
1. Image ID: 32212802240, Number of bbox: 55
2. Image ID: 41822402880, Number of bbox: 55
3. Image ID: 2460640960, Number of bbox: 53
4. Image ID: 8882560320, Number of bbox: 46

Category ID: 9
1. Image ID: 